# Dataset inspection

Raw feature matrices, labels, and sidecars of the six OmicsGBench datasets, read from the Hugging Face revision pinned in `configs/hf/default.yaml` (the one every loader uses).

The matrices are uncorrected: ComBat (AddNeuroMed), covariate regression (MoTrPAC), and the Smoking probe selection are fit by the loader on each training fold, so nothing here is batch- or covariate-adjusted. The first run downloads about 1 GB into the Hugging Face cache (Smoking alone is 644 MB).

In [ ]:
import json
from pathlib import Path
from typing import Any, NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf

import omicsgbench

REPO_ID = 'geometric-intelligence/omicsgbench'
ROOT = Path(omicsgbench.__file__).resolve().parents[1]
REVISION: str = OmegaConf.load(ROOT / 'configs' / 'hf' / 'default.yaml').revision
DATASETS = ['motrpac', 'addneuromed', 'parkinsons', 'brca', 'tuberculosis', 'smoking']
# motrpac_metadata.json has no class mapping; the label is a >15% relative VO2max gain.
MOTRPAC_CLASSES = {'non-responder': 0, 'responder': 1}

sns.set_theme(style='whitegrid', palette='husl')
print(f'{REPO_ID}@{REVISION}')

In [ ]:
def hub_file(filename: str) -> Path:
    return Path(hf_hub_download(REPO_ID, filename, repo_type='dataset', revision=REVISION))


class Summary(NamedTuple):
    stats: dict[str, Any]
    values: np.ndarray
    labels: pd.Series
    feature_mean: np.ndarray
    feature_std: np.ndarray


def summarize(dataset: str, n_values: int = 200_000, seed: int = 0) -> Summary:
    """Load one dataset and keep only what the plots need (Smoking is ~0.5 GB in memory)."""
    values = pd.read_parquet(hub_file(f'{dataset}_data.parquet')).to_numpy(np.float64)
    targets = pd.read_parquet(hub_file(f'{dataset}_targets.parquet'))['target'].to_numpy()
    metadata = json.loads(hub_file(f'{dataset}_metadata.json').read_text())
    mapping = metadata['statistics']['target_stats'].get('class_mapping', MOTRPAC_CLASSES)
    class_names = sorted(mapping, key=mapping.__getitem__)
    n, p = values.shape
    assert len(targets) == n, f'{dataset}: {len(targets)} targets for {n} samples'

    labels = pd.Series(pd.Categorical.from_codes(targets, class_names), name='label')
    counts = labels.value_counts(sort=False)
    finite = values[np.isfinite(values)]
    stats = {
        'samples': n,
        'features': p,
        'n/p': round(n / p, 4),
        'missing %': round(100 * (1 - finite.size / values.size), 3),
        'min': round(float(finite.min()), 3),
        'median': round(float(np.median(finite)), 3),
        'max': round(float(finite.max()), 3),
        'classes': ' / '.join(f'{name} {count}' for name, count in counts.items()),
    }
    rng = np.random.default_rng(seed)
    return Summary(
        stats,
        rng.choice(finite, size=min(n_values, finite.size), replace=False),
        labels,
        np.nanmean(values, axis=0),
        np.nanstd(values, axis=0),
    )


summaries = {dataset: summarize(dataset) for dataset in DATASETS}
pd.DataFrame({dataset: summary.stats for dataset, summary in summaries.items()}).T

## Feature values and class balance

In [ ]:
def facet_grid(title: str) -> tuple[plt.Figure, np.ndarray]:
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    fig.suptitle(title, fontsize=14)
    return fig, axes.flat


fig, axes = facet_grid('Feature values (200k random entries per matrix)')
for ax, (dataset, summary) in zip(axes, summaries.items(), strict=True):
    sns.histplot(summary.values, bins=200, element='step', ax=ax)
    ax.set(title=dataset, xlabel='value', ylabel='count')
fig.tight_layout()

fig, axes = facet_grid('Class balance')
for ax, (dataset, summary) in zip(axes, summaries.items(), strict=True):
    summary.labels.value_counts(sort=False).plot.bar(ax=ax, rot=0)
    ax.set(title=dataset, xlabel='', ylabel='samples')
fig.tight_layout()

fig, axes = facet_grid('Per-feature mean vs SD across samples')
for ax, (dataset, summary) in zip(axes, summaries.items(), strict=True):
    finite = np.isfinite(summary.feature_mean) & np.isfinite(summary.feature_std)
    ax.hexbin(
        summary.feature_mean[finite],
        summary.feature_std[finite],
        gridsize=60,
        bins='log',
        mincnt=1,
    )
    ax.set(title=dataset, xlabel='feature mean', ylabel='feature SD')
fig.tight_layout()

## Batches and covariates vs. label

Sidecar rows are aligned with the data rows. These are the nuisance variables the loader corrects for (AddNeuroMed platform, MoTrPAC covariates) or groups folds by (Tuberculosis country), so how strongly they track the label is worth knowing before reading any result.

In [ ]:
def sidecar(dataset: str, name: str) -> pd.DataFrame:
    frame = pd.read_parquet(hub_file(f'{dataset}_{name}.parquet'))
    labels = summaries[dataset].labels
    assert len(frame) == len(labels), f'{dataset}_{name}: {len(frame)} rows for {len(labels)}'
    return frame.reset_index(drop=True).assign(label=labels)


def by_label(dataset: str, name: str, column: str) -> pd.DataFrame:
    frame = sidecar(dataset, name)
    return pd.crosstab(frame[column], frame['label'], margins=True)


display(by_label('tuberculosis', 'batches', 'batch'))
display(by_label('addneuromed', 'batches', 'batch'))
display(by_label('parkinsons', 'sample_meta', 'batch'))

covariates = sidecar('motrpac', 'covariates')
display(covariates.groupby('label', observed=True).mean(numeric_only=True).round(2))
display(pd.crosstab(covariates['sex'], covariates['label'], margins=True))
display(pd.crosstab(covariates['race'], covariates['label'], margins=True))